# Interactive LightRAG 10-K Query System

This notebook provides an interactive interface for querying 10-K financial documents using LightRAG with:
- **Embedding Model**: BGE-large-en-v1.5 (local)
- **LLM**: Mistral-8b (API)
- **Reranker**: ColBERT small-v1 (local server)
- **Chunking**: 1200 tokens, 100 overlap
- **Query Mode**: Hybrid (KG + vector search)

## Features
- ✅ Persistent chunk storage (no re-chunking)
- ✅ Cached embeddings (no re-computation)
- ✅ Interactive querying with context visualization
- ✅ Before/after reranking comparison
- ✅ Markdown export of results

## Cell 1: Imports & Configuration

In [ ]:
# Standard library imports
import os
import json
import numpy as np
import asyncio
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from functools import partial

# Third-party imports
from dotenv import load_dotenv
from tqdm import tqdm
import nest_asyncio
from sentence_transformers import SentenceTransformer
import torch

# LightRAG imports
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc
from lightrag.rerank import cohere_rerank

# Enable nested asyncio for Jupyter
nest_asyncio.apply()

# Load environment variables
load_dotenv()

# ============================================
# GPU/CPU Configuration
# ============================================
USE_GPU = True  # Set to False to force CPU mode
DEVICE = None

if USE_GPU and torch.cuda.is_available():
    DEVICE = "cuda"
    print("="*60)
    print("🚀 GPU MODE ENABLED")
    print("="*60)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("="*60 + "\n")
elif USE_GPU:
    print("="*60)
    print("⚠ GPU REQUESTED BUT NOT AVAILABLE")
    print("="*60)
    print("Falling back to CPU mode\n")
    DEVICE = "cpu"
else:
    print("="*60)
    print("💻 CPU MODE ENABLED")
    print("="*60 + "\n")
    DEVICE = "cpu"

# Configuration Constants
PARSED_DOCS_PATH = "parsed_10k_documents.json"
CHUNKS_PATH = "chunks_10k.jsonl"
EMBEDDINGS_PATH = "embeddings_10k.npy"
CHUNK_METADATA_PATH = "chunk_metadata.json"
WORKING_DIR = "./lightrag_10k_workspace"
QUERY_RESULTS_DIR = "./query_results"

# Chunking parameters (LightRAG defaults)
CHUNK_TOKEN_SIZE = 1200
CHUNK_OVERLAP_TOKEN_SIZE = 100

# Model parameters
BGE_MODEL_NAME = "BAAI/bge-large-en-v1.5"
EMBEDDING_DIM = 1024
MISTRAL_MODEL = "ministral-8b-latest"
MISTRAL_BASE_URL = "https://api.mistral.ai/v1"

# Create output directories
Path(QUERY_RESULTS_DIR).mkdir(exist_ok=True)

print("✓ Configuration loaded successfully")
print(f"  - Device: {DEVICE.upper()}")
print(f"  - Working directory: {WORKING_DIR}")
print(f"  - Chunk size: {CHUNK_TOKEN_SIZE} tokens")
print(f"  - Overlap: {CHUNK_OVERLAP_TOKEN_SIZE} tokens")
print(f"  - Embedding model: {BGE_MODEL_NAME}")

## Cell 2: Helper Functions

In [ ]:
# ============================================
# Document Loading Functions
# ============================================

def load_parsed_documents(path: str = PARSED_DOCS_PATH) -> Dict:
    """Load parsed 10-K documents from JSON file."""
    print(f"Loading documents from {path}...")
    with open(path, 'r', encoding='utf-8') as f:
        docs = json.load(f)
    print(f"✓ Loaded {len(docs)} companies")
    return docs


# ============================================
# Chunking Functions
# ============================================

def chunk_documents(
    documents: Dict,
    tokenizer,
    chunk_size: int = CHUNK_TOKEN_SIZE,
    overlap: int = CHUNK_OVERLAP_TOKEN_SIZE
) -> List[Dict]:
    """Chunk documents using tokenizer with overlap.
    
    Args:
        documents: Dict of {ticker: {text, metadata}}
        tokenizer: Tokenizer with encode/decode methods
        chunk_size: Maximum tokens per chunk
        overlap: Overlapping tokens between chunks
        
    Returns:
        List of chunk dicts with text and metadata
    """
    chunks = []
    chunk_id = 0
    
    print(f"\nChunking {len(documents)} documents...")
    print(f"  Chunk size: {chunk_size} tokens")
    print(f"  Overlap: {overlap} tokens\n")
    
    for ticker, doc_data in tqdm(documents.items(), desc="Chunking"):
        text = doc_data['text']
        
        # Tokenize full document
        tokens = tokenizer.encode(text)
        
        # Create overlapping chunks
        start = 0
        doc_chunk_idx = 0
        
        while start < len(tokens):
            end = min(start + chunk_size, len(tokens))
            chunk_tokens = tokens[start:end]
            chunk_text = tokenizer.decode(chunk_tokens)
            
            # Create chunk object with provenance
            chunk_obj = {
                'chunk_id': f"chunk_{chunk_id:06d}",
                'text': chunk_text,
                'metadata': {
                    'ticker': ticker,
                    'company_name': doc_data.get('company_name', ''),
                    'period_end_date': doc_data.get('period_end_date', ''),
                    'source_file': doc_data.get('source_file', ''),
                    'doc_chunk_index': doc_chunk_idx,
                    'token_count': len(chunk_tokens),
                    'char_count': len(chunk_text)
                }
            }
            
            chunks.append(chunk_obj)
            chunk_id += 1
            doc_chunk_idx += 1
            
            # Move to next chunk with overlap
            start += chunk_size - overlap
            
            # Break if we've reached the end
            if end >= len(tokens):
                break
    
    print(f"\n✓ Created {len(chunks)} chunks")
    print(f"  Avg chunks per document: {len(chunks) / len(documents):.1f}")
    
    return chunks


def save_chunks(chunks: List[Dict], output_path: str = CHUNKS_PATH):
    """Save chunks to JSONL file."""
    print(f"\nSaving {len(chunks)} chunks to {output_path}...")
    with open(output_path, 'w', encoding='utf-8') as f:
        for chunk in chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + '\n')
    
    file_size = Path(output_path).stat().st_size / (1024 * 1024)  # MB
    print(f"✓ Chunks saved ({file_size:.1f} MB)")


def load_chunks(chunks_path: str = CHUNKS_PATH) -> List[Dict]:
    """Load pre-chunked documents from JSONL file."""
    print(f"Loading chunks from {chunks_path}...")
    chunks = []
    with open(chunks_path, 'r', encoding='utf-8') as f:
        for line in f:
            chunks.append(json.loads(line))
    print(f"✓ Loaded {len(chunks)} chunks")
    return chunks


# ============================================
# Embedding Functions
# ============================================

def compute_embeddings(
    chunks: List[Dict],
    embed_model: SentenceTransformer,
    batch_size: int = 32
) -> np.ndarray:
    """Generate embeddings for all chunks.
    
    Args:
        chunks: List of chunk dicts with 'text' field
        embed_model: SentenceTransformer model instance
        batch_size: Batch size for encoding
        
    Returns:
        Numpy array of embeddings (n_chunks, embedding_dim)
    """
    texts = [chunk['text'] for chunk in chunks]
    
    print(f"\nComputing embeddings for {len(texts)} chunks...")
    print(f"  Model: {embed_model}")
    print(f"  Batch size: {batch_size}\n")
    
    embeddings = embed_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    
    print(f"\n✓ Embeddings computed: {embeddings.shape}")
    return embeddings


def save_embeddings(
    embeddings: np.ndarray,
    chunks: List[Dict],
    embeddings_path: str = EMBEDDINGS_PATH,
    metadata_path: str = CHUNK_METADATA_PATH
):
    """Save embeddings and metadata to files."""
    print(f"\nSaving embeddings and metadata...")
    
    # Save embeddings as numpy array
    np.save(embeddings_path, embeddings)
    emb_size = Path(embeddings_path).stat().st_size / (1024 * 1024)  # MB
    print(f"✓ Embeddings saved to {embeddings_path} ({emb_size:.1f} MB)")
    
    # Save metadata mapping
    metadata = {
        'embedding_dim': embeddings.shape[1],
        'num_chunks': len(chunks),
        'chunk_ids': [chunk['chunk_id'] for chunk in chunks],
        'metadata': {chunk['chunk_id']: chunk['metadata'] for chunk in chunks}
    }
    
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    meta_size = Path(metadata_path).stat().st_size / (1024 * 1024)  # MB
    print(f"✓ Metadata saved to {metadata_path} ({meta_size:.1f} MB)")


def load_embeddings(
    embeddings_path: str = EMBEDDINGS_PATH,
    metadata_path: str = CHUNK_METADATA_PATH
) -> Tuple[np.ndarray, Dict]:
    """Load cached embeddings and metadata.
    
    Returns:
        Tuple of (embeddings array, metadata dict)
    """
    print(f"\nLoading cached embeddings...")
    
    embeddings = np.load(embeddings_path)
    print(f"✓ Embeddings loaded: {embeddings.shape}")
    
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    print(f"✓ Metadata loaded: {metadata['num_chunks']} chunks")
    
    return embeddings, metadata


# ============================================
# Results Formatting Functions
# ============================================

def format_results_to_markdown(
    query: str,
    results: Dict,
    timestamp: str,
    output_dir: str = QUERY_RESULTS_DIR
) -> str:
    """Format query results to markdown file.
    
    Args:
        query: Original query text
        results: Dict containing retrieval results and metadata
        timestamp: Timestamp string for filename
        output_dir: Directory to save markdown file
        
    Returns:
        Path to saved markdown file
    """
    filename = f"query_results_{timestamp}.md"
    filepath = Path(output_dir) / filename
    
    with open(filepath, 'w', encoding='utf-8') as f:
        # Header
        f.write(f"# Query Results\n\n")
        f.write(f"**Query:** {query}\n\n")
        f.write(f"**Timestamp:** {timestamp}\n\n")
        f.write(f"**Mode:** {results.get('mode', 'N/A')}\n\n")
        f.write(f"**Parameters:**\n")
        f.write(f"- Top-k: {results.get('top_k', 'N/A')}\n")
        f.write(f"- Chunk Top-k: {results.get('chunk_top_k', 'N/A')}\n")
        f.write(f"- Reranking: {results.get('reranking_enabled', False)}\n\n")
        f.write("---\n\n")
        
        # Pre-reranking results
        if 'pre_rerank_chunks' in results:
            f.write("## Pre-Reranking Results\n\n")
            for i, chunk in enumerate(results['pre_rerank_chunks'], 1):
                f.write(f"### Chunk {i}\n")
                f.write(f"**Score:** {chunk.get('score', 'N/A'):.4f}\n\n")
                f.write(f"**Ticker:** {chunk.get('ticker', 'N/A')}\n\n")
                f.write(f"**Company:** {chunk.get('company_name', 'N/A')}\n\n")
                f.write(f"**Text:**\n```\n{chunk.get('text', '')}\n```\n\n")
            f.write("\n---\n\n")
        
        # Post-reranking results
        if 'post_rerank_chunks' in results:
            f.write("## Post-Reranking Results\n\n")
            for i, chunk in enumerate(results['post_rerank_chunks'], 1):
                f.write(f"### Chunk {i}\n")
                f.write(f"**Rerank Score:** {chunk.get('rerank_score', 'N/A'):.4f}\n\n")
                f.write(f"**Original Score:** {chunk.get('original_score', 'N/A'):.4f}\n\n")
                f.write(f"**Ticker:** {chunk.get('ticker', 'N/A')}\n\n")
                f.write(f"**Company:** {chunk.get('company_name', 'N/A')}\n\n")
                f.write(f"**Text:**\n```\n{chunk.get('text', '')}\n```\n\n")
            f.write("\n---\n\n")
        
        # Reranking impact analysis
        if 'reranking_impact' in results:
            f.write("## Reranking Impact Analysis\n\n")
            impact = results['reranking_impact']
            f.write(f"- Chunks reordered: {impact.get('chunks_reordered', 0)}\n")
            f.write(f"- Average score change: {impact.get('avg_score_change', 0):.4f}\n")
            f.write(f"- Max position change: {impact.get('max_position_change', 0)}\n\n")
            f.write("---\n\n")
        
        # LLM Answer (if provided)
        if 'llm_answer' in results:
            f.write("## LLM Generated Answer\n\n")
            f.write(f"{results['llm_answer']}\n\n")
    
    return str(filepath)


print("\n✓ Helper functions loaded successfully")

## Cell 3: Model Initialization

In [ ]:
print("Initializing models...\n")

# ============================================
# 1. Load BGE-Large Embedding Model (Local)
# ============================================
print(f"Loading {BGE_MODEL_NAME} on {DEVICE.upper()}...")
bge_model = SentenceTransformer(BGE_MODEL_NAME, device=DEVICE)
print(f"✓ BGE-Large loaded")
print(f"  - Dimension: {EMBEDDING_DIM}")
print(f"  - Device: {bge_model.device}\n")

# Test embedding
test_embed = bge_model.encode(["test"], normalize_embeddings=True)
print(f"  Test embedding shape: {test_embed.shape}")
assert test_embed.shape[1] == EMBEDDING_DIM, f"Expected dim {EMBEDDING_DIM}, got {test_embed.shape[1]}"
print("  ✓ Embedding test passed\n")


# ============================================
# 2. Setup Mistral LLM Wrapper (API)
# ============================================
async def mistral_llm_func(
    prompt,
    system_prompt=None,
    history_messages=[],
    **kwargs
) -> str:
    """Mistral LLM wrapper using OpenAI-compatible API."""
    return await openai_complete_if_cache(
        MISTRAL_MODEL,
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=os.getenv("MISTRAL_API_KEY"),
        base_url=MISTRAL_BASE_URL,
        **kwargs
    )

print(f"✓ Mistral LLM configured ({MISTRAL_MODEL})\n")


# ============================================
# 3. Setup BGE Embedding Function for LightRAG
# ============================================
async def bge_embedding_func(texts: List[str]) -> np.ndarray:
    """BGE embedding wrapper for LightRAG async interface."""
    return bge_model.encode(texts, normalize_embeddings=True)

print("✓ BGE embedding function configured\n")


# ============================================
# 4. Setup ColBERT Reranker (Local Server)
# ============================================
rerank_func = partial(
    cohere_rerank,
    model=os.getenv("RERANK_MODEL", "answerdotai/answerai-colbert-small-v1"),
    api_key=os.getenv("RERANK_BINDING_API_KEY", "<not-needed>"),
    base_url=os.getenv("RERANK_BINDING_HOST", "http://localhost:7997/v1/rerank"),
    enable_chunking=False,
    max_tokens_per_doc=512  # ColBERT small context window limit
)

print(f"✓ ColBERT reranker configured")
print(f"  - Model: {os.getenv('RERANK_MODEL')}")
print(f"  - Endpoint: {os.getenv('RERANK_BINDING_HOST')}")
print(f"  - Max tokens per doc: 512\n")

# Test reranker connection
try:
    test_docs = ["test document 1", "test document 2"]
    test_results = await rerank_func(
        query="test query",
        documents=test_docs,
        top_n=2
    )
    print("  ✓ Reranker connection test passed\n")
except Exception as e:
    print(f"  ⚠ Reranker connection test failed: {e}")
    print("  Continuing anyway... reranking may not work\n")

print("="*60)
print("✓ All models initialized successfully")
print("="*60)

## Cell 4: Data Processing Pipeline

In [ ]:
import tiktoken

print("Starting data processing pipeline...\n")
print("="*60)

# ============================================
# Step 4.1: Load or Create Chunks
# ============================================
chunks = None

if Path(CHUNKS_PATH).exists():
    print("Found existing chunks file!")
    user_choice = input(f"Load existing chunks from {CHUNKS_PATH}? (y/n): ").lower()
    
    if user_choice == 'y':
        chunks = load_chunks(CHUNKS_PATH)
    else:
        print("Will create new chunks...\n")

if chunks is None:
    print("Creating new chunks...\n")
    
    # Load parsed documents
    documents = load_parsed_documents(PARSED_DOCS_PATH)
    
    # Initialize tokenizer (using tiktoken for consistency with LightRAG)
    tokenizer = tiktoken.get_encoding("cl100k_base")
    print(f"✓ Tokenizer initialized: {tokenizer.name}\n")
    
    # Chunk documents
    chunks = chunk_documents(
        documents,
        tokenizer,
        chunk_size=CHUNK_TOKEN_SIZE,
        overlap=CHUNK_OVERLAP_TOKEN_SIZE
    )
    
    # Save chunks
    save_chunks(chunks, CHUNKS_PATH)

print("\n" + "="*60)
print(f"Chunks ready: {len(chunks)} total")
print("="*60 + "\n")


# ============================================
# Step 4.2: Load or Create Embeddings
# ============================================
embeddings = None
chunk_metadata = None

if Path(EMBEDDINGS_PATH).exists() and Path(CHUNK_METADATA_PATH).exists():
    print("Found existing embeddings!")
    user_choice = input(f"Load existing embeddings from {EMBEDDINGS_PATH}? (y/n): ").lower()
    
    if user_choice == 'y':
        embeddings, chunk_metadata = load_embeddings(
            EMBEDDINGS_PATH,
            CHUNK_METADATA_PATH
        )
    else:
        print("Will compute new embeddings...\n")

if embeddings is None:
    print("Computing new embeddings...\n")
    
    # Compute embeddings
    embeddings = compute_embeddings(chunks, bge_model, batch_size=32)
    
    # Save embeddings and metadata
    save_embeddings(embeddings, chunks, EMBEDDINGS_PATH, CHUNK_METADATA_PATH)
    
    # Load metadata back for consistency
    with open(CHUNK_METADATA_PATH, 'r') as f:
        chunk_metadata = json.load(f)

print("\n" + "="*60)
print("Embeddings ready!")
print("="*60 + "\n")


# ============================================
# Step 4.3: Display Statistics
# ============================================
print("📊 Data Statistics:")
print("="*60)
print(f"Total chunks: {len(chunks):,}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Embedding shape: {embeddings.shape}")

# Calculate storage sizes
chunks_size = Path(CHUNKS_PATH).stat().st_size / (1024 * 1024) if Path(CHUNKS_PATH).exists() else 0
embeddings_size = Path(EMBEDDINGS_PATH).stat().st_size / (1024 * 1024) if Path(EMBEDDINGS_PATH).exists() else 0
metadata_size = Path(CHUNK_METADATA_PATH).stat().st_size / (1024 * 1024) if Path(CHUNK_METADATA_PATH).exists() else 0

print(f"\nStorage:")
print(f"  Chunks: {chunks_size:.1f} MB")
print(f"  Embeddings: {embeddings_size:.1f} MB")
print(f"  Metadata: {metadata_size:.1f} MB")
print(f"  Total: {chunks_size + embeddings_size + metadata_size:.1f} MB")

# Get unique tickers
unique_tickers = set(chunk['metadata']['ticker'] for chunk in chunks)
print(f"\nDocuments:")
print(f"  Unique companies: {len(unique_tickers)}")
print(f"  Avg chunks per company: {len(chunks) / len(unique_tickers):.1f}")

# Token statistics
token_counts = [chunk['metadata']['token_count'] for chunk in chunks]
print(f"\nChunk tokens:")
print(f"  Min: {min(token_counts)}")
print(f"  Max: {max(token_counts)}")
print(f"  Avg: {np.mean(token_counts):.0f}")

print("="*60)
print("✓ Data processing pipeline complete!")
print("="*60)

## Cell 5: LightRAG Initialization

In [ ]:
import shutil

print("Initializing LightRAG...\n")
print("="*60)

# ============================================
# Clean up old workspace if exists
# ============================================
if Path(WORKING_DIR).exists():
    print(f"Found existing workspace: {WORKING_DIR}")
    user_choice = input("Delete existing workspace and start fresh? (y/n): ").lower()
    
    if user_choice == 'y':
        print("Deleting old workspace...")
        shutil.rmtree(WORKING_DIR)
        print(f"✓ Workspace deleted\n")
    else:
        print("Keeping existing workspace\n")

# Create workspace directory
Path(WORKING_DIR).mkdir(exist_ok=True)


# ============================================
# Initialize LightRAG Instance
# ============================================
print("Creating LightRAG instance...\n")

rag = LightRAG(
    working_dir=WORKING_DIR,
    
    # LLM Configuration
    llm_model_func=mistral_llm_func,
    llm_model_name=MISTRAL_MODEL,
    
    # Embedding Configuration
    embedding_func=EmbeddingFunc(
        embedding_dim=EMBEDDING_DIM,
        max_token_size=8192,
        func=bge_embedding_func
    ),
    
    # Chunking Configuration (matches our pre-chunking)
    chunk_token_size=CHUNK_TOKEN_SIZE,
    chunk_overlap_token_size=CHUNK_OVERLAP_TOKEN_SIZE,
    
    # Reranker Configuration
    rerank_model_func=rerank_func
)

print("Initializing storages...")
await rag.initialize_storages()

print("\n" + "="*60)
print("✓ LightRAG initialized successfully!")
print("="*60)
print(f"\nConfiguration:")
print(f"  Working dir: {WORKING_DIR}")
print(f"  LLM: {MISTRAL_MODEL}")
print(f"  Embedding: {BGE_MODEL_NAME}")
print(f"  Reranker: {os.getenv('RERANK_MODEL')}")
print(f"  Chunk size: {CHUNK_TOKEN_SIZE} tokens")
print(f"  Overlap: {CHUNK_OVERLAP_TOKEN_SIZE} tokens")
print("="*60)

## Cell 6: Index Documents into LightRAG

In [ ]:
print("Indexing documents into LightRAG...\n")
print("="*60)

# Check if already indexed
doc_status_file = Path(WORKING_DIR) / "kv_store_doc_status.json"
already_indexed = doc_status_file.exists()

if already_indexed:
    with open(doc_status_file, 'r') as f:
        doc_status = json.load(f)
    
    print(f"Found existing index with {len(doc_status)} documents")
    user_choice = input("Skip indexing and use existing data? (y/n): ").lower()
    
    if user_choice == 'y':
        print("✓ Using existing index\n")
        already_indexed = True
    else:
        print("Will re-index documents...\n")
        already_indexed = False

if not already_indexed:
    print(f"Indexing {len(chunks)} chunks...\n")
    print("This may take a while (entity extraction, graph building, etc.)\n")
    
    # Group chunks by ticker for better organization
    ticker_chunks = {}
    for chunk in chunks:
        ticker = chunk['metadata']['ticker']
        if ticker not in ticker_chunks:
            ticker_chunks[ticker] = []
        ticker_chunks[ticker].append(chunk['text'])
    
    print(f"Grouped into {len(ticker_chunks)} companies\n")
    
    # Insert documents by company
    for i, (ticker, texts) in enumerate(tqdm(ticker_chunks.items(), desc="Indexing companies"), 1):
        # Combine all chunks for this company into one document
        combined_text = "\n\n".join(texts)
        
        # Insert into LightRAG
        await rag.ainsert(combined_text)
        
        # Progress update every 10 companies
        if i % 10 == 0:
            print(f"\nIndexed {i}/{len(ticker_chunks)} companies...")
    
    print("\n" + "="*60)
    print("✓ Indexing complete!")
    print("="*60)
    
    # Show storage statistics
    workspace_size = sum(f.stat().st_size for f in Path(WORKING_DIR).rglob('*') if f.is_file())
    workspace_size_mb = workspace_size / (1024 * 1024)
    
    print(f"\nWorkspace statistics:")
    print(f"  Total size: {workspace_size_mb:.1f} MB")
    print(f"  Location: {WORKING_DIR}")
    
    # List key files
    print(f"\nGenerated files:")
    for file in sorted(Path(WORKING_DIR).iterdir()):
        if file.is_file():
            size = file.stat().st_size / 1024  # KB
            print(f"  - {file.name}: {size:.1f} KB")

print("\n" + "="*60)
print("Ready for querying!")
print("="*60)

## Cell 7: Interactive Query Interface

In [ ]:
# ============================================
# Interactive Query Function
# ============================================

async def query_lightrag(
    question: str,
    context_only: bool = True,
    mode: str = "hybrid",
    top_k: int = 60,
    chunk_top_k: int = 20,
    enable_rerank: bool = True,
    save_to_file: bool = True
) -> Dict:
    """Query LightRAG and return structured results.
    
    Args:
        question: Query text
        context_only: If True, only retrieve context (no LLM answer)
        mode: Query mode (hybrid, local, global, naive, mix)
        top_k: Number of entities/relations to retrieve
        chunk_top_k: Number of chunks to retrieve
        enable_rerank: Enable reranking of chunks
        save_to_file: Save results to markdown file
        
    Returns:
        Dict with query results and metadata
    """
    print("\n" + "="*60)
    print("🔍 Query Processing")
    print("="*60)
    print(f"Question: {question}")
    print(f"Mode: {mode}")
    print(f"Context only: {context_only}")
    print(f"Reranking: {enable_rerank}")
    print("="*60 + "\n")
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Query with context retrieval
    print("Retrieving context...")
    
    # First get context without reranking to compare
    context_pre_rerank = await rag.aquery(
        question,
        param=QueryParam(
            mode=mode,
            only_need_context=True,
            top_k=top_k,
            chunk_top_k=chunk_top_k,
            enable_rerank=False  # Disable for comparison
        )
    )
    
    # Parse pre-rerank context
    # Note: LightRAG returns context as formatted string
    # We'll need to parse it to extract individual chunks
    pre_rerank_chunks = parse_context_string(context_pre_rerank)
    print(f"✓ Retrieved {len(pre_rerank_chunks)} chunks (pre-rerank)\n")
    
    # Query with reranking if enabled
    post_rerank_chunks = []
    llm_answer = None
    
    if enable_rerank or not context_only:
        print("Querying with reranking...")
        
        result = await rag.aquery(
            question,
            param=QueryParam(
                mode=mode,
                only_need_context=context_only,
                top_k=top_k,
                chunk_top_k=chunk_top_k,
                enable_rerank=enable_rerank
            )
        )
        
        if context_only:
            # Parse reranked context
            post_rerank_chunks = parse_context_string(result)
            print(f"✓ Retrieved {len(post_rerank_chunks)} chunks (post-rerank)\n")
        else:
            # Result contains the LLM answer
            llm_answer = result
            print(f"✓ Generated answer ({len(llm_answer)} chars)\n")
    
    # Calculate reranking impact
    reranking_impact = None
    if enable_rerank and post_rerank_chunks:
        reranking_impact = calculate_reranking_impact(
            pre_rerank_chunks,
            post_rerank_chunks
        )
    
    # Compile results
    results = {
        'query': question,
        'timestamp': timestamp,
        'mode': mode,
        'top_k': top_k,
        'chunk_top_k': chunk_top_k,
        'reranking_enabled': enable_rerank,
        'context_only': context_only,
        'pre_rerank_chunks': pre_rerank_chunks,
        'post_rerank_chunks': post_rerank_chunks,
        'reranking_impact': reranking_impact,
        'llm_answer': llm_answer
    }
    
    # Save to file if requested
    if save_to_file:
        filepath = format_results_to_markdown(question, results, timestamp)
        print(f"💾 Results saved to: {filepath}\n")
    
    return results


def parse_context_string(context: str) -> List[Dict]:
    """Parse LightRAG context string into structured chunks.
    
    LightRAG returns context as formatted text. This function
    attempts to parse it into individual chunks with metadata.
    """
    # This is a simplified parser - may need adjustment based on actual format
    chunks = []
    
    # Split by common separators
    sections = context.split("\n\n")
    
    for i, section in enumerate(sections):
        if section.strip():
            chunks.append({
                'text': section.strip(),
                'score': 1.0 - (i * 0.01),  # Placeholder score
                'ticker': 'N/A',  # Would need to extract from text
                'company_name': 'N/A'
            })
    
    return chunks


def calculate_reranking_impact(
    pre_rerank: List[Dict],
    post_rerank: List[Dict]
) -> Dict:
    """Calculate metrics showing impact of reranking."""
    # Calculate position changes
    position_changes = []
    
    for i, post_chunk in enumerate(post_rerank):
        # Find original position
        for j, pre_chunk in enumerate(pre_rerank):
            if pre_chunk['text'] == post_chunk['text']:
                position_changes.append(abs(i - j))
                break
    
    return {
        'chunks_reordered': len([p for p in position_changes if p > 0]),
        'max_position_change': max(position_changes) if position_changes else 0,
        'avg_position_change': np.mean(position_changes) if position_changes else 0,
        'avg_score_change': 0  # Placeholder
    }


print("✓ Query functions loaded")
print("\nReady for interactive queries!")
print("\nUsage:")
print("  results = await query_lightrag(\"your question here\")")
print("  results = await query_lightrag(\"your question\", context_only=False)  # Get LLM answer")

## Cell 8: Example Queries & Results Visualization

In [ ]:
# ============================================
# Example Query 1: Context Only
# ============================================

question = "What are Apple's main revenue sources in 2024?"

print("\n🎯 Example Query: Context Only Mode")
print("="*60)

results = await query_lightrag(
    question=question,
    context_only=True,  # Only retrieve context, no LLM answer
    mode="hybrid",
    enable_rerank=True,
    save_to_file=True
)

# Display results
print("\n📋 Pre-Reranking Results:")
print("="*60)
for i, chunk in enumerate(results['pre_rerank_chunks'][:3], 1):  # Show top 3
    print(f"\n{i}. Score: {chunk['score']:.4f}")
    print(f"   Ticker: {chunk['ticker']}")
    print(f"   Text preview: {chunk['text'][:200]}...")

if results['post_rerank_chunks']:
    print("\n\n📋 Post-Reranking Results:")
    print("="*60)
    for i, chunk in enumerate(results['post_rerank_chunks'][:3], 1):  # Show top 3
        print(f"\n{i}. Rerank Score: {chunk.get('rerank_score', chunk['score']):.4f}")
        print(f"   Original Score: {chunk['score']:.4f}")
        print(f"   Ticker: {chunk['ticker']}")
        print(f"   Text preview: {chunk['text'][:200]}...")

if results['reranking_impact']:
    print("\n\n📊 Reranking Impact:")
    print("="*60)
    impact = results['reranking_impact']
    print(f"Chunks reordered: {impact['chunks_reordered']}")
    print(f"Max position change: {impact['max_position_change']}")
    print(f"Avg position change: {impact['avg_position_change']:.2f}")

print("\n" + "="*60)
print("✓ Query complete!")
print("="*60)

## Interactive Query Cell (Run Multiple Times)

In [ ]:
# ============================================
# Interactive Query Interface
# ============================================

print("\n" + "="*60)
print("🎯 Interactive Query Mode")
print("="*60)
print("\nEnter your question below:")

# Get user input
question = input("\nQuestion: ")

if not question.strip():
    print("❌ Empty question. Please try again.")
else:
    # Configuration
    print("\nConfiguration:")
    context_only_input = input("Context only? (y/n, default: y): ").lower()
    context_only = context_only_input != 'n'
    
    mode_input = input("Query mode (hybrid/local/global/naive/mix, default: hybrid): ").lower()
    mode = mode_input if mode_input in ['hybrid', 'local', 'global', 'naive', 'mix'] else 'hybrid'
    
    enable_rerank_input = input("Enable reranking? (y/n, default: y): ").lower()
    enable_rerank = enable_rerank_input != 'n'
    
    # Execute query
    results = await query_lightrag(
        question=question,
        context_only=context_only,
        mode=mode,
        enable_rerank=enable_rerank,
        save_to_file=True
    )
    
    # Display results
    print("\n" + "="*60)
    print("📊 RESULTS")
    print("="*60)
    
    if context_only:
        # Show retrieved chunks
        print("\n🔍 Retrieved Context (Top 5):")
        print("-"*60)
        
        display_chunks = results['post_rerank_chunks'] if results['post_rerank_chunks'] else results['pre_rerank_chunks']
        
        for i, chunk in enumerate(display_chunks[:5], 1):
            print(f"\n[{i}] Ticker: {chunk['ticker']} | Company: {chunk['company_name']}")
            print(f"    Score: {chunk.get('rerank_score', chunk['score']):.4f}")
            print(f"    Text: {chunk['text'][:300]}...")
            print("-"*60)
        
        # Show reranking impact if available
        if results['reranking_impact']:
            print("\n📈 Reranking Impact:")
            print("-"*60)
            impact = results['reranking_impact']
            print(f"  Chunks reordered: {impact['chunks_reordered']}")
            print(f"  Max position change: {impact['max_position_change']}")
            print(f"  Avg position change: {impact['avg_position_change']:.2f}")
    
    else:
        # Show LLM answer
        print("\n💬 LLM Answer:")
        print("-"*60)
        print(results['llm_answer'])
        print("-"*60)
    
    print("\n✓ Results saved to markdown file")
    print(f"  Location: {QUERY_RESULTS_DIR}/query_results_{results['timestamp']}.md")
    print("\n" + "="*60)